# 神经元的发放频率

用漏积分发放（LIF）模型看两件事：外部突触输入变强时，神经元每秒发放多少次；如果神经元之间再接上兴奋性回返突触，这条频率曲线会怎样变，网络还会不会稳住。

单位：时间用 ms，电压用 mV，电流用 pA，电导用 nS，电容用 pF。膜时间常数 \(\tau_m = C / g_L\)。

## 0. 准备

导入数值和绘图库。配色按由浅到深排（下标 0 最浅），后面把不同参数的曲线画在同一张图上时用来区分。

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
from scipy.sparse import csr_matrix
from tqdm.auto import tqdm, trange

# 配色由浅到深，下标 0 最浅
ColorBlue = [
    "#00B4FA",
    "#00A3E9",
    "#0093D9",
    "#0082C8",
    "#0071B7",
    "#0061A7",
    "#005096",
    "#003F85",
    "#002F75",
    "#001E64",
]
ColorRed = [
    "#FA5050",
    "#F24747",
    "#EA3E3E",
    "#E33535",
    "#DB2C2C",
    "#D32424",
    "#CB1B1B",
    "#C41212",
    "#BC0909",
    "#B40000",
]
ColorGreen = [
    "#96D250",
    "#85C850",
    "#75BE50",
    "#64B450",
    "#53AA50",
    "#43A050",
    "#329650",
    "#218C50",
    "#118250",
    "#007850",
]
ColorYellow = [
    "#FAFA00",
    "#FAE903",
    "#FAD907",
    "#FAC80A",
    "#FAB70D",
    "#FAA711",
    "#FA9614",
    "#FA8517",
    "#FA751B",
    "#FA641E",
]
ColorPurple = [
    "#FA82FA",
    "#E774E8",
    "#D465D6",
    "#C157C5",
    "#AE48B3",
    "#9C3AA1",
    "#892B8F",
    "#761D7E",
    "#630E6C",
    "#50005A",
]
ColorGray = [
    "#B4B4B4",
    "#A4A4A4",
    "#959595",
    "#858585",
    "#767676",
    "#666666",
    "#575757",
    "#474747",
    "#383838",
    "#282828",
]

## 1. LIF

漏积分发放：膜电位向静息电位指数衰减，外部电流把它往上推。一旦越过阈值就记一次发放，电位回到复位值，并在不应期内忽略输入电流。

### 1.1 准备

全笔记共用的时间步长和 LIF 参数。膜时间常数 \(\tau_m = C / g_L = 20\,\mathrm{ms}\)。

In [ ]:
dt = 0.01 # ms
Time = 1000.0 # ms
T = int(Time/dt)  # 1 s 的时间步数

# tau_m = C / g_L = 20 ms
V_rest = -70.0 # 静息电位，mV
V_reset = -60.0 # 复位电位，mV
threshold = -50.0 # 阈值，mV
t_ref = 2 # 不应期，ms
ref_length = int(t_ref/dt)  # 不应期的时间步数
C = 500.0 # 膜电容，pF
g_L = 25.0 # 漏电导，nS

os.makedirs("pics", exist_ok=True)

### 1.2 LIF模型

给单个神经元一段恒定电流，用欧拉法积分膜方程，看它会不会周期发放。这里还没有突触，输入只有外加电流 \(I_\mathrm{ext}\)。

In [ ]:
# 单个神经元的电流钳，只跑 200 ms
Time_exp = 200.0 # ms
T_exp = int(Time_exp / dt)

# 提前设置默认电压为静息电位
V = np.full(T_exp, V_rest)  # mV

# 不应期内为 0，用来关掉输入电流
# 这个数组用来存储是否在不应期内。乘上他的值就可以在不应期内屏蔽外部电流。
not_refractory = np.ones(T_exp)

spikes = np.zeros(T_exp)

# 设置外部电流。
I_ext = np.zeros(T_exp)  # 外加电流，pA
t_start_pulse = 50 # ms
t_end_pulse = 150 # ms
# 600 pA 的无阈值稳态是 V_rest + I/g_L = -46 mV，高于阈值，所以会反复发放
I_ext[int(t_start_pulse / dt):int(t_end_pulse / dt)] = 600

# 开始实验
for t in trange(1, T_exp):
    # 这里的trange不需要感到奇怪。这就是一个进度条。改成range效果是一样的，只是没有进度条而已。
    # C dV/dt = -g_L (V - V_rest) + I_ext * not_refractory

    # 可以试着写一下
    # - 用欧拉法积分一下
    # - 别忘了怎么加入不应期
    # - 核心是记录spikes，但是spike也是要画的。
    # - 不应期应该避免溢出。
    # - numpy其实可以用bool作为index来筛选。
    # TODO
    pass


fig, axs = plt.subplots(3, 1, figsize=(6, 10), constrained_layout=True)

axs[0].set_title("V of neuron")
axs[0].set_xlabel("Time (ms)")
axs[0].set_ylabel("Voltage (mV)")
axs[0].set_xlim((0, Time_exp))
axs[1].set_title("I_ext of neuron")
axs[1].set_xlabel("Time (ms)")
axs[1].set_ylabel("Current (pA)")
axs[1].set_xlim((0, Time_exp))
axs[2].set_title("Total spikes")
axs[2].set_xlabel("Time (ms)")
axs[2].set_ylabel("neuron")
axs[2].set_xlim((0, Time_exp))

fig.suptitle(
    "LIF Model"
)


t = np.arange(T_exp) * dt
axs[0].plot(t, V, label="V of Neuron")
axs[1].plot(t, I_ext, label="I_ext of Neuron")

axs[2].scatter(np.where(spikes == 1)[0] * dt, np.zeros_like(np.where(spikes == 1)[0] * dt))

axs[0].legend()
axs[1].legend()
plt.savefig("pics/lif.png", dpi=300)
plt.show()

## 2. AMPA

把方波电流换成 AMPA 型兴奋性突触电流。每个输入事件把门控变量 \(s\) 抬高 1，随后 \(s\) 以时间常数 \(\tau_\mathrm{AMPA}\) 指数衰减。电导电流是 \(I = g_\mathrm{AMPA}\, s\, (V - E)\)，反转电位 \(E = 0\,\mathrm{mV}\)，所以在阈下它是去极化的。

### 2.1 发放

先只接外部泊松输入，回返连接概率设为 0。每个时间步、每个神经元以概率 \(\nu_\mathrm{ext}\Delta t\) 收到一个外部脉冲（\(\nu_\mathrm{ext}\) 用 kHz，\(\Delta t\) 用 ms，乘积就是这一步的事件概率）。

In [ ]:
"""
外部 AMPA，没有回返连接。

I = g_ampa * (V_E - V) * s，V_E = 0
ds/dt = -s / tau_ampa + 外部脉冲
电流存成正值，膜方程用 +I/C。
"""

g_ampa = 2.0 # nS
tau_ampa = 2.0 # ms
V_E = 0.0 # mV，兴奋性反转电位
N = 1000
rng = np.random.default_rng()

# 这里要开始多维度并行计算了。
V = np.full((N, T), V_rest)  # mV，行是神经元，列是时间，下同

not_refractory = np.ones((N, T))
spikes = np.zeros((N, T))

I_ext = np.zeros((N, T))  # 外部电流，pA

S_ampa_ext = np.zeros((N, T))

synapse_ratio = 0.00  # 回返连接概率

noise_freq = 2.1 # 外部噪声频率，kHz。可以试着改变一下。

# 每步事件概率。kHz * ms 无量纲，2.1 * 0.01 = 0.021
p_noise = noise_freq * dt

spikes_external = rng.binomial(1, p_noise, (N, T))


for t in trange(1, T):
    # 积分可以延续上面的，但是这一次要积分S了。
    # I = g_ampa * (V_E - V) * s，V_E = 0
    # ds/dt = -s / tau_ampa + delta(t - t_ext)
    # 而且由于并行，index可以写作[:, t]这样。
    # TODO
    pass


fig, axs = plt.subplots(3, 1, figsize=(6, 10), constrained_layout=True)

axs[0].set_title("V of neurons 0-4")
axs[0].set_xlabel("Time (ms)")
axs[0].set_ylabel("Voltage (mV)")
axs[1].set_title("I_ext of neurons 0-4")
axs[1].set_xlabel("Time (ms)")
axs[1].set_ylabel("Current (pA)")
axs[2].set_title("Total spikes")
axs[2].set_xlabel("Time (ms)")
axs[2].set_ylabel("neuron")

fig.suptitle(
    f"LIF Model with noise frequency {noise_freq} KHz and synapse ratio {synapse_ratio}"
)

# 电压和电流只画前 5 个；栅格图是全体
for idx in range(5):
    t = np.arange(T) * dt
    axs[0].plot(t, V[idx, :], label=f"V of Neuron {idx}")
    axs[1].plot(t, I_ext[idx, :], label=f"I_ext of Neuron {idx}")


axs[2].scatter(np.where(spikes == 1)[1] * dt, np.where(spikes == 1)[0], s=0.01)

axs[0].legend()
axs[1].legend()

plt.savefig(f"pics/lif_{noise_freq}KHz_synapse{synapse_ratio}.png", dpi=300)
plt.show()

### 2.2 在不同的频率下运行

扫描外部噪声频率，记录群体平均发放率，得到没有回返连接时的输入–输出曲线。这一格最耗时，扫完会把结果写到 `data/`，下一格可以直接读回来重画。

In [ ]:
"""
扫描外部泊松频率，记录群体平均发放率。p = 0，没有回返。

从这里起电流改成 g * (V - V_E) * s（兴奋性时为负），膜方程减去它，效果仍是去极化。
"""

noise_freqs = np.arange(1.5, 3.05, 0.05)  # kHz
spike_freqs = np.zeros_like(noise_freqs) # Hz

# 这一次需要对多个频率进行并行计算。可以用一个for循环来做。
# 内部就是之前的代码。
# 不过，这次的目的是记录发放率和噪声频率的关系。
# 并行运算1000个神经元，然后取平均，可以减少误差。
# 所以前面的并行计算依旧是有用的。
# TODO

# 存下来，画图时不用重跑
os.makedirs("data", exist_ok=True)
np.save("data/noise_freqs.npy", noise_freqs)
np.save("data/spike_freqs.npy", spike_freqs)


反复运行上面的代码很耗时，所以扫描结束时把横轴和纵轴存成 `data/noise_freqs.npy`（外部噪声频率，kHz）和 `data/spike_freqs.npy`（群体平均发放率，Hz）。重画图时运行下面两格即可，不必重跑仿真。

In [ ]:
# 上一格存的曲线。noise_freqs：kHz，spike_freqs：Hz
noise_freqs: np.typing.NDArray = np.load("data/noise_freqs.npy")
spike_freqs: np.typing.NDArray = np.load("data/spike_freqs.npy")

In [ ]:
# 无回返连接时的输入-输出曲线
plt.title("Spike Frequency at different noise frequencies")
plt.xlabel("Noise Frequency (KHz)")
plt.ylabel("Spike Frequency (Hz)")
plt.plot(noise_freqs, spike_freqs, color=ColorRed[6])

## 3. 如果加入神经元内连接呢？

在外部 AMPA 之外，再让神经元彼此以概率 \(p\) 建立兴奋性突触，平均入度是 \(Np\)。一个神经元发放会抬高其他神经元的 \(s\)，从而抬高群体发放率。下面用一条过零点的直线估计这条抬升：发放率（Hz）约为 \((
u_\mathrm{ext} - z) \times 1000 / (Np)\)。\(z\) 是直线和横轴的交点，取在无连接曲线仍然很低的 2.1 kHz 附近。

### 3.1 发放

和 2.1 同一套积分，但 `synapse_ratio > 0`，回返电流 \(I_\mathrm{rec}\) 加进膜方程。

In [ ]:
"""
在 2.1 上加回返 AMPA。W 为 0/1 稀疏矩阵，平均入度 N * synapse_ratio。
I_rec = g_ampa * (V_E - V) * (W @ s)，s 按突触前神经元索引。
"""

g_ampa = 2.0 # nS
tau_ampa = 2.0 # ms
V_E = 0.0 # mV
N = 1000
rng = np.random.default_rng()


V = np.full((N, T), V_rest)  # mV

not_refractory = np.ones((N, T))
spikes = np.zeros((N, T))

I_ext = np.zeros((N, T))  # 外部电流，pA
I_rec = np.zeros((N, T))  # 回返电流，pA

S_ampa_ext = np.zeros((N, T))  # 外部脉冲驱动
S_ampa_rec = np.zeros((N, T))  # 该神经元自己的发放驱动

synapse_ratio = 0.015  # 回返连接概率，平均入度约 15。可以改一下。

noise_freq = 2.1 # 外部噪声频率，kHz。可以改一下看看有什么区别。

p_noise = noise_freq * dt

spikes_external = rng.binomial(1, p_noise, (N, T))

# 核心出装
# W[i, j] = 1 表示 j 连到 i
# 就用这个矩阵来选择其他神经元的输入spike吧
W = csr_matrix(rng.random((N, N)) < synapse_ratio)


# 依旧是可以沿袭前面的积分
# 不过这次，不要忘了输入电流是两部分了，一个是I_ext，一个是I_rec。
# 而且，这次需要再积分一个S，是Recurrent的S。
# 还要考虑用W选择一些连接。
# TODO

fig, axs = plt.subplots(3, 1, figsize=(6, 10), constrained_layout=True)

axs[0].set_title("V of neurons 0-4")
axs[0].set_xlabel("Time (ms)")
axs[0].set_ylabel("Voltage (mV)")
axs[1].set_title("I_ext of neurons 0-4")
axs[1].set_xlabel("Time (ms)")
axs[1].set_ylabel("Current (pA)")
axs[2].set_title("Total spikes")
axs[2].set_xlabel("Time (ms)")
axs[2].set_ylabel("neuron")

fig.suptitle(
    f"LIF Model with noise frequency {noise_freq} KHz and synapse ratio {synapse_ratio}"
)

for idx in range(5):
    t = np.arange(T) * dt
    axs[0].plot(t, V[idx, :], label=f"V of Neuron {idx}")
    axs[1].plot(t, I_ext[idx, :], label=f"I_ext of Neuron {idx}")  # 只画外部电流


axs[2].scatter(np.where(spikes == 1)[1] * dt, np.where(spikes == 1)[0], s=0.01)

axs[0].legend()
axs[1].legend()

plt.savefig(f"pics/lif_{noise_freq}KHz_synapse{synapse_ratio}.png", dpi=300)
plt.show()

### 3.2 理论预测

红色曲线是 2.2 节仿真得到的无连接响应。彩色斜线是上面的平均场直线：斜率由 \(N\) 和 \(p\) 决定，零点由所选的 \(z\) 决定。水平虚线是若干 \((z, p)\) 组合下、从仿真里读到的发放率，用来和斜线对照。斜线本身在 \(z\) 处穿过 0，并不穿过这些水平线。

In [ ]:
plt.xlim(1.5, 3)
plt.ylim(0, 50)
plt.xlabel("Noise Frequency (KHz)")
plt.ylabel("Spike Frequency (Hz)")

# 红线：p = 0 的仿真曲线
plt.plot(noise_freqs, spike_freqs, label="Response Curve", color=ColorRed[6])
def line(x: float, z: float, synapse_ratio: float, neuron_count: int) -> float:
    # r = (x - z) * 1000 / (N * p)，单位 Hz。x、z 为 kHz，N*p 是平均入度
    return (x-z) * 1000 / (neuron_count * synapse_ratio)


plt.plot(noise_freqs, line(noise_freqs, 2.1, 0.015, N), label="z=2.1khz, p=0.015", color=ColorYellow[6])
plt.plot(noise_freqs, line(noise_freqs, 2.1, 0.02, N), label="z=2.1khz, p=0.02", color=ColorYellow[9])
# p = 0 时公式除以零，改成竖线画在 z
plt.vlines(2.1, 0, 50, label="z=2.1khz, p=0", linestyles="--", color=ColorYellow[3])

# 灰线是仿真里读到的发放率，不是 line() 的值
plt.hlines(5, 1.5, 3, colors=ColorGray[3], linestyles="--", label="Spike Freq at z=2.1khz, p=0")
plt.hlines(10, 1.5, 3, colors=ColorGray[6], linestyles="--", label="Spike Freq at z=2.1khz, p=0.015")
plt.hlines(20, 1.5, 3, colors=ColorGray[9], linestyles="--", label="Spike Freq at z=2.1khz, p=0.02")
plt.title("Predicted Frequencies")
plt.legend()

In [ ]:
# 这里可以多plot一些。这里是一些例子。
# 把零点 z 改到 2.0 和 2.2 kHz，斜率不变
plt.title("Some more predictions")
plt.xlim(1.5, 3)
plt.ylim(0, 50)
plt.xlabel("Noise Frequency (KHz)")
plt.ylabel("Spike Frequency (Hz)")

plt.plot(noise_freqs, spike_freqs, label="Response Curve", color=ColorRed[6])

plt.plot(noise_freqs, line(noise_freqs, 2.2, 0.015, N), label="z=2.2khz, p=0.015", color=ColorGreen[6])
plt.plot(noise_freqs, line(noise_freqs, 2.2, 0.02, N), label="z=2.2khz, p=0.02", color=ColorGreen[9])
plt.plot(noise_freqs, line(noise_freqs, 2.0, 0.015, N), label="z=2.0khz, p=0.015", color=ColorPurple[6])
plt.plot(noise_freqs, line(noise_freqs, 2.0, 0.02, N), label="z=2.0khz, p=0.02", color=ColorPurple[9])

plt.hlines(20, 1.5, 3, colors=ColorGray[6], linestyles="--", label="Spike Freq at z=2.2khz, p=0.015")
plt.hlines(38.5, 1.5, 3, colors=ColorGray[9], linestyles="--", label="Spike Freq at z=2.2khz, p=0.015")
plt.hlines(5, 1.5, 3, colors=ColorGray[3], linestyles="--", label="Spike Freq at z=2.0khz, p=0.02")
plt.hlines(3.75, 1.5, 3, colors=ColorGray[2], linestyles="--", label="Spike Freq at z=2.0khz, p=0.015")

plt.legend()

### 3.3 实验验证

外部频率固定，重复多次，用 50 ms 滑窗（每 5 ms 移动一次）估计发放率随时间的变化。连接概率先取 0：恒定输入下，这条曲线应在短暂过渡之后保持平稳。

In [ ]:
# 固定外部频率、p = 0，重复 10 次取平均
Rep = 10

noise_freq = 2.1

p_noise = noise_freq * dt
p = 0.0


# Sliding window
# 50 ms 窗，每 5 ms 移动一次。x 是窗口中心
window_size = int(50 / dt)
shift_size = int(5 / dt)
x = np.arange(window_size // 2, T - window_size // 2, shift_size)
spike_freqs_exp = np.zeros((Rep, len(x)))

# 在这里可以用sliding window来画出其变化
# 接下来请计算发放率的变化吧
# 可以试着做10次取平均。
# TODO

plt.plot(x, np.mean(spike_freqs_exp, axis=0))
plt.xlim(0, T)
# 横轴刻度从时间步换回 ms
plt.xticks(np.arange(0, T + 1, int(T / 10)), np.arange(0, 1001, 100))
plt.xlabel("Time (ms)")
plt.ylabel("Spike Frequency (Hz)")
plt.title(f"Spike Frequency vs Time, freq={p_noise / dt}KHz, p={p}")
plt.savefig(f"pics/spike_freq_{noise_freq}khz_synapse_{p}.png", dpi=300)

## 4. 稳定性测试

回返连接打开（\(p = 0.015\)）。外部频率大部分时间停在 2.1 kHz，在 300–500 ms 阶跃到 2.2 kHz，再回到 2.1 kHz。看群体发放率是否跟着阶跃走、阶跃结束后能否回到原来的水平，而不是被兴奋性回返推着一路升高。

In [ ]:
# 回返网络对外部频率阶跃的响应，10 次平均
Rep = 10

noise_freq = 2.1  # 基线，kHz
test_noise_freq = 2.2  # 阶跃期间，kHz

# 现在p_noise不再是一个常数，而是一个数组了
# 请尝试在中间改一下噪声频率
# 下面是一个例子。
p_noise = np.full((T, ), noise_freq * dt)
t_start_pulse = 300  # ms
t_end_pulse = 500  # ms
p_noise[int(t_start_pulse / dt) : int(t_end_pulse / dt)] = test_noise_freq * dt

# 连接
p = 0.015

window_size = int(50 / dt)
shift_size = int(5 / dt)
x = np.arange(window_size // 2, T - window_size // 2, shift_size)
spike_freqs_exp = np.zeros((Rep, len(x)))

# 延续上面观察发放率变化的代码。
# TODO